In [ ]:
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from shapely.geometry import box
from pathlib import Path


DEM_PATH = Path("../data/intermediate/geographic/merged_dem.tif")

In [ ]:
def merge_dem_files(input_paths: list[str], output_path: str):
    """
    Merge multiple DEM files into a single TIF file

    Args:
        input_paths: List of paths to the DEM files to be merged
        output_path: Where to save the merged file
    """
    # List to store open dataset objects
    src_files_to_mosaic = []

    # Open all files
    for dem_file in input_paths:
        src = rasterio.open(dem_file)
        src_files_to_mosaic.append(src)

    # Merge the files
    mosaic, out_trans = merge(src_files_to_mosaic)

    # Copy the metadata from the first file
    out_meta = src_files_to_mosaic[0].meta.copy()

    # Update the metadata
    out_meta.update(
        {
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
        }
    )

    # Write the mosaic to disk
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    # Close all the files
    for src in src_files_to_mosaic:
        src.close()

    return output_path


dem_files = [
    DEM_PATH / "10_DEM_y40x10/10_DEM_y40x10.tif",
    DEM_PATH / "10_DEM_y40x20/10_DEM_y40x20.tif",
    DEM_PATH / "10_DEM_y50x10/10_DEM_y50x10.tif",
    DEM_PATH / "10_DEM_y50x20/10_DEM_y50x20.tif",
]


# # Usage example:
merged_dem_path = merge_dem_files(dem_files, "../data/intermediate/geographic/merged_dem.tif")

In [ ]:
# Define Poland's extreme points
POLAND_BOUNDS = {
    "north": 54.835784,  # Jastrzębia Góra
    "south": 49.002167,  # Opołonek
    "west": 14.123333,  # Osinów Dolny
    "east": 24.145556,  # Zosin
}


def clip_dem_to_poland(input_path, output_path):
    """
    Clip DEM file to Poland's boundaries and save as new file
    """
    # Create Poland's bounding box
    poland_geometry = [
        box(
            POLAND_BOUNDS["west"] * 0.9,
            POLAND_BOUNDS["south"] * 0.9,
            POLAND_BOUNDS["east"] * 1.1,
            POLAND_BOUNDS["north"] * 1.1,
        )
    ]

    with rasterio.open(input_path) as src:
        # Perform the clipping
        out_image, out_transform = mask(src, poland_geometry, crop=True)

        # Update the metadata
        out_meta = src.meta.copy()
        out_meta.update(
            {"height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform}
        )

        # Save the clipped raster
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_image)


# Paths
input_dem = "../data/input/geographic/merged_dem.tif"
output_dem = "../data/intermediate/geographic/poland_dem.tif"

# Create the clipped DEM
clip_dem_to_poland(input_dem, output_dem)